# CUAD → Legal-BERT Fine-Tuning Pipeline

Thin orchestrator. All logic lives in the `legal_ner` package; each cell just calls a stage.

**Before running:** set the runtime to **Python 3.10 + T4 GPU** (`Runtime → Change runtime type`).

Run order: setup → download → convert → chunk+tokenize → train → evaluate → inference.

## 1. Environment setup

Installs the pinned 2024 cohort. **After this cell finishes you MUST restart the runtime once** (`Runtime → Restart session`), then run from the next cell down — this ensures the pinned numpy/tokenizers are the ones imported.

In [ ]:
# Pinned cohort. torch is left to Colab's CUDA-matched build (see requirements.txt).
!pip install -q \
    transformers==4.46.0 \
    datasets==3.0.1 \
    tokenizers==0.20.1 \
    huggingface_hub==0.26.2 \
    accelerate==1.0.1 \
    evaluate==0.4.3 \
    seqeval==1.2.2 \
    "numpy>=1.26,<2.0"
print("Install done. Now: Runtime → Restart session, then continue from the next cell.")

In [ ]:
# --- Run everything below AFTER restarting the runtime ---
import sys
assert sys.version_info[:2] == (3, 10), f"Expected Python 3.10, got {sys.version.split()[0]}"

import torch
assert torch.cuda.is_available(), "GPU not enabled — Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

import transformers, datasets
print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets   :", datasets.__version__)
transformers.set_seed(42)

In [ ]:
# Mount Drive and point the package at a Drive-backed project dir BEFORE importing legal_ner
# (config.PROJECT_DIR reads LEGAL_NER_PROJECT_DIR at import time).
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['LEGAL_NER_PROJECT_DIR'] = '/content/drive/MyDrive/legal_ner_project'
os.makedirs(os.environ['LEGAL_NER_PROJECT_DIR'], exist_ok=True)

# Clone the PUBLIC repo into Drive on first run; pull the latest on subsequent runs.
# No credentials needed because the repo is public. The clone persists in Drive across
# sessions, so after the first time this is just a fast `git pull`.
REPO_URL = 'https://github.com/sukesh46/legal-bert-finetuning.git'
REPO_DIR = os.path.join(os.environ['LEGAL_NER_PROJECT_DIR'], 'repo')
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    !git -C "$REPO_DIR" pull --ff-only
else:
    !git clone "$REPO_URL" "$REPO_DIR"

# Put the freshly-pulled src/ on the import path.
src_path = os.path.join(REPO_DIR, 'src')
assert os.path.isdir(src_path), f'src/ not found under {REPO_DIR} — did the clone succeed?'
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Capture the exact resolved environment for reproducibility.
!pip freeze > "$LEGAL_NER_PROJECT_DIR/requirements.lock.txt"
print('Project dir:', os.environ['LEGAL_NER_PROJECT_DIR'])
print('Repo       :', REPO_DIR)

## 2. Download + validate CUAD

Set `limit=None` for the full corpus; a small int (e.g. 5) is handy for a quick smoke run.

In [ ]:
from legal_ner import download_cuad
cuad, stats = download_cuad.download_cuad(limit=None)
print(download_cuad.format_stats(stats))

## 3. Convert CUAD → BIO, validate reconstruction, split at contract level

The reconstruction check is acceptance criterion 1 — it must report **no mismatches**.

In [ ]:
from legal_ner import cuad_to_bio

rows, conv_stats = cuad_to_bio.convert_cuad(cuad)
print(f"Converted {conv_stats.contracts} contracts, {conv_stats.qa_pairs_used} QA pairs used, "
      f"{conv_stats.impossible_skipped} impossible skipped.")
if conv_stats.overlap_overrides:
    print('Top overlap overrides (Strategy A):')
    for pair, n in sorted(conv_stats.overlap_overrides.items(), key=lambda x: -x[1])[:10]:
        print(f'  {pair}: {n}')

# Acceptance criterion 1: span reconstruction on a 20-example sample.
mismatches = cuad_to_bio.validate_reconstruction(cuad, rows, sample_size=20)
assert not mismatches, f"Span reconstruction FAILED: {mismatches[:3]}"
print('✓ Span reconstruction passed (acceptance criterion 1).')

splits = cuad_to_bio.contract_level_split(rows)
print({k: len(v) for k, v in splits.items()})

## 4. Chunk long contracts + tokenize/align, build HF datasets

In [ ]:
from datasets import Dataset
from legal_ner import chunking, tokenize_align, labels as lbl
from legal_ner.train import load_model_and_tokenizer

label2id, id2label = lbl.build_label_maps()
lbl.save_labels(os.path.join(os.environ['LEGAL_NER_PROJECT_DIR'], 'labels.json'))

_, tokenizer = load_model_and_tokenizer()

def build_split(split_rows):
    chunked = chunking.chunk_rows(split_rows)
    ds = Dataset.from_list(chunked)
    return ds.map(
        lambda ex: tokenize_align.tokenize_and_align_labels(ex, tokenizer, label2id),
        batched=True,
        remove_columns=ds.column_names,
    )

tok_train = build_split(splits['train'])
tok_val   = build_split(splits['val'])
tok_test  = build_split(splits['test'])
print('tokenized sizes:', len(tok_train), len(tok_val), len(tok_test))

## 5. Train (resumes from a Drive checkpoint if one exists)

In [ ]:
from legal_ner import train
trainer = train.train(tok_train, tok_val)

## 6. Evaluate on the test split (aggregate + per-category report)

In [ ]:
from legal_ner import evaluate

preds = trainer.predict(tok_test)
metrics = evaluate.make_compute_metrics(id2label)((preds.predictions, preds.label_ids))
print('Aggregate:', metrics)

true_labels, pred_labels = evaluate.decode_predictions(preds.predictions, preds.label_ids, id2label)
support = evaluate.training_support(tok_train, id2label)
print(evaluate.per_category_report(true_labels, pred_labels, support=support))

## 7. Inference on a raw contract

In [ ]:
from legal_ner import inference

final_dir = os.path.join(os.environ['LEGAL_NER_PROJECT_DIR'], 'checkpoints', 'final')
model, tok, id2label_loaded = inference.load_for_inference(final_dir)

sample_text = "This Agreement shall be governed by the laws of the State of Delaware."
spans = inference.predict(sample_text, model, tok, id2label_loaded)
for s in spans:
    print(f"[{s.category}] ({s.start_char}:{s.end_char}) {s.text!r}")